# Claims Reserving Using the Chain Ladder Method

This notebook applies the Chain Ladder method to estimate cumulative development factors, ultimate claims, and IBNR (Incurred But Not Reported) reserves.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
from google.colab import files

uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_excel(file_name)

df.head()

In [ ]:
# Pastikan nama kolom sesuai data
df.columns = [
    "Development",
    "Claim Lag",
    "Claim",
    "Premium",
    "Customer Claim",
    "Customer Premium"
]

# Rapikan index
df = df.reset_index(drop=True)

# Pastikan Claim Lag berupa numerik
df["Claim Lag"] = pd.to_numeric(df["Claim Lag"], errors="coerce")

# Pastikan Claim berupa numerik
df["Claim"] = pd.to_numeric(df["Claim"], errors="coerce")

# Hapus baris yang tidak memiliki informasi penting
df = df.dropna(subset=["Development", "Claim Lag", "Claim"])

print("Ukuran data:", df.shape)
df.head()

In [ ]:
# Pastikan index rapi
df = df.reset_index(drop=True)

# Pisahkan tahun dan kuartal
df["Dev_Year"] = df["Development"].str[:4].astype(int)
df["Dev_Quarter"] = df["Development"].str[-1].astype(int)

# Indeks periode development
df["Development_Index"] = (
    df["Dev_Year"] * 4 + df["Dev_Quarter"]
)

# Indeks Origin
df["Origin_Index"] = (
    df["Development_Index"] - df["Claim Lag"]
)

# Kembalikan ke tahun dan kuartal
df["Origin_Year"] = (
    (df["Origin_Index"] - 1) // 4
)

df["Origin_Quarter"] = (
    (df["Origin_Index"] - 1) % 4 + 1
)

# Bentuk Origin
df["Origin"] = (
    df["Origin_Year"].astype(str)
    + " Q"
    + df["Origin_Quarter"].astype(str)
)

# Cek hasil
df[[
    "Development",
    "Claim Lag",
    "Claim",
    "Origin"
]].head(20)

In [ ]:
incremental_triangle = df.pivot(
    index="Origin",
    columns="Claim Lag",
    values="Claim"
)

# Pastikan urutan Origin dan Claim Lag benar
incremental_triangle = incremental_triangle.sort_index()
incremental_triangle = incremental_triangle.sort_index(axis=1)

print("Ukuran triangle:", incremental_triangle.shape)

incremental_triangle

In [ ]:
incremental_triangle_na = incremental_triangle.copy()

# Periode development terakhir yang tersedia
last_year = 2023
last_quarter = 2

# Loop setiap Origin dan Claim Lag
for i in range(incremental_triangle_na.shape[0]):

    origin_text = incremental_triangle_na.index[i]

    # Ambil tahun dan kuartal Origin
    origin_year = int(origin_text[:4])
    origin_quarter = int(origin_text[-1])

    for j in range(incremental_triangle_na.shape[1]):

        lag = int(incremental_triangle_na.columns[j])

        # Hitung tahun dan kuartal Development
        total_quarters = (
            origin_quarter - 1 + lag
        )

        development_year = (
            origin_year + total_quarters // 4
        )

        development_quarter = (
            total_quarters % 4 + 1
        )

        # Jika melewati 2023 Q2,
        # maka data belum observed
        if (
            development_year > last_year
            or (
                development_year == last_year
                and development_quarter > last_quarter
            )
        ):
            incremental_triangle_na.iloc[i, j] = np.nan

print("Ukuran triangle:", incremental_triangle_na.shape)
print(
    "Jumlah NA:",
    incremental_triangle_na.isna().sum().sum()
)

incremental_triangle_na

In [ ]:
cumulative_triangle = incremental_triangle_na.cumsum(axis=1)

cumulative_triangle

In [ ]:
development_factors = []

for j in range(cumulative_triangle.shape[1] - 1):

    current = cumulative_triangle.iloc[:, j]
    next_value = cumulative_triangle.iloc[:, j + 1]

    # Hanya gunakan pasangan yang tersedia
    valid = (
        current.notna()
        & next_value.notna()
        & (current != 0)
    )

    numerator = next_value[valid].sum()
    denominator = current[valid].sum()

    factor = numerator / denominator

    development_factors.append(factor)

development_factor_table = pd.DataFrame({
    "Claim Lag": range(len(development_factors)),
    "Development Factor": development_factors
})

development_factor_table

In [ ]:
cdf = []

for i in range(len(development_factors)):

    future_factors = development_factors[i:]

    cdf_value = np.prod(future_factors)

    cdf.append(cdf_value)

# Lag terakhir tidak membutuhkan development factor lagi
cdf.append(1.0)

cdf_table = pd.DataFrame({
    "Claim Lag": range(len(cdf)),
    "Development Factor": development_factors + [np.nan],
    "CDF": cdf
})

cdf_table

In [ ]:
latest_cumulative = cumulative_triangle.apply(
    lambda row: row.dropna().iloc[-1],
    axis=1
)

latest_lag = cumulative_triangle.apply(
    lambda row: row.dropna().index[-1],
    axis=1
)

latest_table = pd.DataFrame({
    "Latest Lag": latest_lag.astype(int),
    "Reported Claim": latest_cumulative
})

latest_table

In [ ]:
latest_table["CDF"] = latest_table["Latest Lag"].apply(
    lambda lag: cdf[int(lag)]
)

latest_table

In [ ]:
latest_table["Ultimate Claim"] = (
    latest_table["Reported Claim"]
    * latest_table["CDF"]
)

latest_table

In [ ]:
latest_table["IBNR"] = (
    latest_table["Ultimate Claim"]
    - latest_table["Reported Claim"]
)

latest_table

In [ ]:
result_df = latest_table.reset_index()

result_df = result_df.rename(
    columns={"index": "Origin"}
)

result_df = result_df[[
    "Origin",
    "Latest Lag",
    "Reported Claim",
    "CDF",
    "Ultimate Claim",
    "IBNR"
]]

result_df

In [ ]:
result_display = result_df.copy()

for col in [
    "Reported Claim",
    "Ultimate Claim",
    "IBNR"
]:
    result_display[col] = result_display[col].map(
        lambda x: f"{x:,.2f}"
    )

result_display["CDF"] = result_display["CDF"].map(
    lambda x: f"{x:.6f}"
)

result_display

In [ ]:
total_reported = result_df["Reported Claim"].sum()
total_ultimate = result_df["Ultimate Claim"].sum()
total_ibnr = result_df["IBNR"].sum()

print(f"Total Reported Claim : {total_reported:,.2f}")
print(f"Total Ultimate Claim : {total_ultimate:,.2f}")
print(f"Total IBNR           : {total_ibnr:,.2f}")

In [ ]:
# Cek hubungan Ultimate = Reported + IBNR
check = (
    result_df["Reported Claim"]
    + result_df["IBNR"]
    - result_df["Ultimate Claim"]
)

print("Maksimum selisih perhitungan:",
      abs(check).max())

In [ ]:
plt.figure(figsize=(14, 9))

sns.heatmap(
    incremental_triangle_na,
    annot=True,
    fmt=".0f",
    cmap="Blues",
    linewidths=0.5,
    cbar_kws={"label": "Claim"}
)

plt.title("Incremental Claims Development Triangle", fontsize=16, fontweight="bold")
plt.xlabel("Claim Lag")
plt.ylabel("Origin Period")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 9))

sns.heatmap(
    cumulative_triangle,
    annot=True,
    fmt=".0f",
    cmap="Greens",
    linewidths=0.5,
    cbar_kws={"label": "Cumulative Claim"}
)

plt.title("Cumulative Claims Development Triangle", fontsize=16, fontweight="bold")
plt.xlabel("Claim Lag")
plt.ylabel("Origin Period")
plt.tight_layout()
plt.show()

In [ ]:
# Cek panjang Development Factor dan CDF
print("Jumlah Development Factor :", len(development_factors))
print("Jumlah CDF                 :", len(cdf))

In [ ]:
# Samakan panjang berdasarkan jumlah data yang tersedia
n = min(len(development_factors), len(cdf))

development_factor_df = pd.DataFrame({
    "Claim Lag": range(n),
    "Development Factor": development_factors[:n],
    "CDF": cdf[:n]
})

development_factor_df

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    development_factor_df["Claim Lag"],
    development_factor_df["Development Factor"],
    marker="o"
)

plt.xlabel("Claim Lag")
plt.ylabel("Development Factor")
plt.title("Development Factor per Claim Lag")
plt.xticks(development_factor_df["Claim Lag"])
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    development_factor_df["Claim Lag"],
    development_factor_df["CDF"],
    marker="o"
)

plt.xlabel("Claim Lag")
plt.ylabel("CDF")
plt.title("Cumulative Development Factor (CDF)")
plt.xticks(development_factor_df["Claim Lag"])
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(
    result_df["Origin"],
    result_df["IBNR"]
)

plt.xlabel("Origin Period")
plt.ylabel("IBNR")
plt.title("IBNR per Origin Period")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    result_df["Origin"],
    result_df["Reported Claim"],
    marker="o",
    label="Reported Claim"
)

plt.plot(
    result_df["Origin"],
    result_df["Ultimate Claim"],
    marker="o",
    label="Ultimate Claim"
)

plt.xlabel("Origin Period")
plt.ylabel("Claim Amount")
plt.title("Reported Claim vs Ultimate Claim")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 50)
print("RINGKASAN HASIL CHAIN LADDER")
print("=" * 50)

print(f"Total Reported Claim : {total_reported:,.2f}")
print(f"Total Ultimate Claim : {total_ultimate:,.2f}")
print(f"Total IBNR           : {total_ibnr:,.2f}")

print(
    f"\nIBNR terhadap Ultimate Claim : "
    f"{total_ibnr / total_ultimate * 100:.2f}%"
)